# BP/OSD cross-validation

This notebook tunes `decode_bp_osd` on a deterministic subset of the labelled training data. Each candidate is evaluated separately for the `X` and `Z` bases using cross-validation, and the selected configuration minimizes the combined logical error rate (LER). Runtime is reported as a secondary metric.

The folds are used only for evaluation: BP/OSD itself is not trained on the fold data.

In [23]:
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path("..")
REPO_ROOT = REPO_ROOT.resolve()
requirements_path = REPO_ROOT / "submission_template" / "requirements.txt"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(requirements_path)])
print("installed dependencies for:", sys.executable)


installed dependencies for: /Users/suneg/Dev/QIS-hackathon/qis-my-error/.venv/bin/python



[notice] A new release of pip available: 22.3.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [24]:
import sys
import time
from pathlib import Path

import numpy as np

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "submission_template"))

from hackathon_2026 import load_kaggle_csv, logical_error_rate, problem_info
from bp_osd import build_graph, decode_bp_osd

DATA_PATH = REPO_ROOT / "data" / "train" / "train.csv"
SUBSET_PER_BASIS = 5000
N_FOLDS = 3
SEED = 42
print("repository:", REPO_ROOT.resolve())
print("data:", DATA_PATH)

repository: /Users/suneg/Dev/QIS-hackathon/qis-my-error
data: /Users/suneg/Dev/QIS-hackathon/qis-my-error/data/train/train.csv


## Load the labelled data and basis-specific graphs

In [25]:
def load_basis_data(data_path: str | Path, subset_per_basis: int, seed: int):
    data = load_kaggle_csv(str(data_path))
    if "observable_flips" not in data:
        raise ValueError("cross-validation requires L0-L3 labels")
    result = {}
    for offset, basis in enumerate(("X", "Z")):
        rows = np.flatnonzero(data["basis"] == basis)
        if len(rows) == 0:
            raise ValueError(f"no {basis} rows found")
        rng = np.random.default_rng(seed + offset)
        rng.shuffle(rows)
        if subset_per_basis is not None:
            rows = rows[:min(subset_per_basis, len(rows))]
        result[basis] = {
            "events": data["detector_events"][rows].astype(np.uint8),
            "labels": data["observable_flips"][rows].astype(bool),
        }
    return result

def load_graphs(data_folder: str | Path):
    graphs = {}
    for basis in ("X", "Z"):
        circuit = __import__("stim").Circuit.from_file(str(Path(data_folder) / f"{basis}_circuit.stim"))
        info = problem_info(circuit)
        graphs[basis] = build_graph(info["H"], info["L"], info["p"])
        print(f"{basis}: H={info['H'].shape}, L={info['L'].shape}, p={info['p'].shape}")
    return graphs

basis_data = load_basis_data(DATA_PATH, SUBSET_PER_BASIS, SEED)
graphs = load_graphs(DATA_PATH.parent)
for basis in ("X", "Z"):
    print(basis, basis_data[basis]["events"].shape, basis_data[basis]["labels"].shape)

X: H=(70, 2470), L=(4, 2470), p=(2470,)
Z: H=(70, 2465), L=(4, 2465), p=(2465,)
X (5000, 70) (5000, 4)
Z (5000, 70) (5000, 4)


## Deterministic cross-validation folds

## Candidate configurations and parameter meanings

`decode_bp_osd` has two expensive stages. BP passes messages for `max_iter` iterations. OSD then searches detector-consistent candidates. The grid below is deliberately staged from faster to slower.

- `max_iter`: BP iterations. Higher values allow information to travel farther through the graph, but cost roughly proportionally more BP work and may amplify oscillation.
- `alpha`: normalized-min-sum scale. Values below `1` reduce overconfident messages; `1` is stronger but can be less stable.
- `damping`: retains part of the previous BP message. Higher values are slower to adapt but can stabilize loopy BP.
- `always_osd`: when `False`, valid BP hard decisions skip OSD; when `True`, every shot gets the OSD refinement.
- `avg_post`: gives OSD the accumulated BP beliefs rather than only the last iteration. This often helps when messages oscillate.
- `lam`: number of likely dependent columns considered as OSD single-column alternatives. Larger values search more candidates.
- `lam2`: number of outside columns considered in pair combinations. Pair work grows approximately quadratically as `lam2 * (lam2 - 1) / 2`.
- `coset`: `0` chooses the best individual candidate, `1` chooses the most probable complete logical class, and `2` makes independent bitwise decisions. `2` matches the bitwise LER objective.

`FAST_GRID` is suitable for a larger subset. `BP_GRID` isolates slower or more cautious BP settings. `OSD_GRID` explores larger candidate searches, including configurations with both higher `max_iter` and larger OSD sweeps. Run the groups separately if the full grid is too slow.


In [26]:
from itertools import product

# Broad search around the defaults. This intentionally creates many candidates;
# use a small SUBSET_PER_BASIS first and promote only the best candidates to full data.
BASE = {"max_iter": 3, "alpha": 0.8, "always_osd": True, "damping": 0.5, "avg_post": True, "lam": 400, "lam2": 100, "coset": 2}

# Cheap baselines and early-exit variants.
FAST_GRID = [
    {**BASE, "always_osd": False, "lam": lam, "lam2": lam2}
    for lam, lam2 in ((0, 0), (25, 5), (50, 10), (100, 25), (200, 50), (400, 100))
]

# Broad BP sweep: 288 combinations. Each axis includes values slower or more
# aggressive than the default max_iter=3, alpha=0.8, damping=0.5.
BP_GRID = [
    {**BASE, "max_iter": max_iter, "alpha": alpha, "damping": damping}
    for max_iter, alpha, damping in product(
        (1, 2, 3, 5, 8, 12, 16, 20),
        (0.4, 0.6, 0.7, 0.8, 0.9, 1.0),
        (0.1, 0.25, 0.5, 0.7, 0.85, 0.95),
    )
]

# Compare last-iteration beliefs with averaged beliefs at several iteration counts.
POST_GRID = [
    {**BASE, "max_iter": max_iter, "avg_post": avg_post}
    for max_iter, avg_post in product((1, 2, 3, 5, 8, 12, 20), (False, True))
]

# OSD sweeps. Pair-search cost is approximately lam2*(lam2-1)/2 per shot.
OSD_GRID = [
    {**BASE, "lam": lam, "lam2": lam2}
    for lam, lam2 in (
        (0, 0), (25, 5), (50, 10), (100, 25), (200, 50),
        (300, 75), (400, 50), (400, 100), (400, 150),
        (600, 100), (600, 150), (800, 200), (1000, 250),
    )
]

# Interactions between slower BP and deeper OSD searches.
SLOW_INTERACTIONS = [
    {**BASE, "max_iter": max_iter, "lam": lam, "lam2": lam2}
    for max_iter, lam, lam2 in product(
        (5, 8, 12, 20),
        (400, 600, 800, 1000),
        (100, 150, 200, 250),
    )
    if lam2 <= lam
]

# Compare all logical aggregation modes for representative search depths.
COSET_GRID = [
    {**BASE, "coset": coset, "lam": lam, "lam2": lam2}
    for coset, (lam, lam2) in product(
        (0, 1, 2), ((100, 25), (200, 50), (400, 100), (800, 200))
    )
]

# The expanded grid can be expensive. Deduplicate while preserving cheap-to-expensive order.
PARAM_GRID = []
_seen = set()
for candidate in FAST_GRID + BP_GRID + POST_GRID + OSD_GRID + SLOW_INTERACTIONS + COSET_GRID:
    key = tuple(sorted(candidate.items()))
    if key not in _seen:
        _seen.add(key)
        PARAM_GRID.append(candidate)
print(f"testing {len(PARAM_GRID)} configurations")
print("warning: run this broad grid on a small subset before using the full dataset")


testing 29 configurations


In [27]:
def evaluate_configuration(params: dict) -> dict:
    basis_scores = {"X": [], "Z": []}
    total_wrong = 0
    total_bits = 0
    total_seconds = 0.0
    for basis in ("X", "Z"):
        events = basis_data[basis]["events"]
        labels = basis_data[basis]["labels"]
        for fold_index, test_indices in enumerate(folds[basis]):
            started = time.perf_counter()
            predictions, stats = decode_bp_osd(
                graphs[basis],
                events[test_indices],
                return_stats=True,
                **params,
            )
            elapsed = time.perf_counter() - started
            score = logical_error_rate(predictions, labels[test_indices])
            wrong = int(np.count_nonzero(predictions.astype(bool) != labels[test_indices]))
            total_wrong += wrong
            total_bits += predictions.size
            total_seconds += elapsed
            basis_scores[basis].append({
                "fold": fold_index,
                "ler": score,
                "seconds": elapsed,
                "osd_fraction": float(np.mean(stats == -1)),
            })
    return {
        "params": params,
        "combined_ler": total_wrong / total_bits,
        "seconds": total_seconds,
        "basis": basis_scores,
    }

## Run the search

The objective is the combined LER across all held-out `X` and `Z` folds. Ties are broken by total runtime.

In [28]:
results = []
print(f"evaluating {len(PARAM_GRID)} configurations")
for index, params in enumerate(PARAM_GRID, start=1):
    result = evaluate_configuration(params)
    results.append(result)
    print(
        f"{index}/{len(PARAM_GRID)} "
        f"LER={result['combined_ler']:.6f} "
        f"time={result['seconds']:.2f}s "
        f"params={params}"
    )

results.sort(key=lambda result: (result["combined_ler"], result["seconds"]))
best = results[0]
print("\nbest configuration:", best["params"])
print(f"best combined LER: {best['combined_ler']:.6f}")
print(f"best CV runtime: {best['seconds']:.2f}s")


evaluating 29 configurations
1/29 LER=0.040875 time=0.55s params={'max_iter': 3, 'alpha': 0.8, 'always_osd': False, 'damping': 0.5, 'avg_post': True, 'lam': 50, 'lam2': 10, 'coset': 2}
2/29 LER=0.038300 time=0.51s params={'max_iter': 3, 'alpha': 0.8, 'always_osd': False, 'damping': 0.5, 'avg_post': True, 'lam': 100, 'lam2': 25, 'coset': 2}
3/29 LER=0.038025 time=0.74s params={'max_iter': 3, 'alpha': 0.8, 'always_osd': True, 'damping': 0.5, 'avg_post': True, 'lam': 100, 'lam2': 25, 'coset': 2}
4/29 LER=0.036750 time=0.65s params={'max_iter': 3, 'alpha': 0.8, 'always_osd': True, 'damping': 0.5, 'avg_post': True, 'lam': 200, 'lam2': 50, 'coset': 2}
5/29 LER=0.040000 time=0.79s params={'max_iter': 3, 'alpha': 0.8, 'always_osd': True, 'damping': 0.5, 'avg_post': True, 'lam': 200, 'lam2': 50, 'coset': 0}
6/29 LER=0.039725 time=0.65s params={'max_iter': 3, 'alpha': 0.8, 'always_osd': True, 'damping': 0.5, 'avg_post': True, 'lam': 200, 'lam2': 50, 'coset': 1}
7/29 LER=0.037200 time=0.54s param

In [29]:
for basis in ("X", "Z"):
    print(f"\n{basis} folds")
    for fold_result in best["basis"][basis]:
        print(fold_result)

print("\nUse this dictionary in the final decoder:")
print(best["params"])
print("\nImportant: rerun the selected configuration on a fresh holdout before trusting the result.")



X folds
{'fold': 0, 'ler': 0.03224355128974205, 'seconds': 0.18185683299816446, 'osd_fraction': 1.0}
{'fold': 1, 'ler': 0.03734253149370126, 'seconds': 0.19207387499773176, 'osd_fraction': 1.0}
{'fold': 2, 'ler': 0.03361344537815126, 'seconds': 0.203750749999017, 'osd_fraction': 1.0}

Z folds
{'fold': 0, 'ler': 0.03719256148770246, 'seconds': 0.22028549999959068, 'osd_fraction': 1.0}
{'fold': 1, 'ler': 0.038992201559688064, 'seconds': 0.19518629200319992, 'osd_fraction': 1.0}
{'fold': 2, 'ler': 0.03331332533013205, 'seconds': 0.17923404199973447, 'osd_fraction': 1.0}

Use this dictionary in the final decoder:
{'max_iter': 3, 'alpha': 0.8, 'always_osd': True, 'damping': 0.5, 'avg_post': True, 'lam': 800, 'lam2': 200, 'coset': 2}

Important: rerun the selected configuration on a fresh holdout before trusting the result.
